In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import os

# Bootstrap function

In [2]:
def bootstrap_spearman(x, y, nboot=100, conf=0.95, random_state=None):

    # initiate random number generator
    rng = np.random.default_rng(random_state)
    n = len(x)
    rhos = []

    # calculate rho for each bootstrapped sample
    for i in range(nboot):
        idx = rng.integers(0, n, n)   # sample indices with replacement
        rho, _ = spearmanr(x[idx], y[idx])
        if not np.isnan(rho):
            rhos.append(rho)

    # get rho for the sample
    rho_hat, _ = spearmanr(x, y)

    # calculate confidence intervals
    alpha = (1 - conf) / 2
    ci = np.quantile(rhos, [alpha, 1 - alpha])
    #return {"rho": rho_hat, "ci": ci, "boot_rhos": rhos, "ci_low" : ci[0], "ci_high" : ci[1]}
    return [rho_hat, ci[0], ci[1]]

### Example

In [3]:
dx = np.array([1.2, 2.5, 3.0, 4.1, 5.3])  # your data (n=5)
dy = np.array([2.1, 2.8, 0.9, 4.0, 2.8])

In [4]:
dboot = bootstrap_spearman(dx, dy, nboot=50, conf=0.95, random_state=15)
#dboot['rho'], dboot['ci_low'], dboot['ci_high']
dboot

[0.46169025843831935, -0.6666666666666666, 1.0]

# DATA

In [5]:
df = pd.read_csv("01B_garden_compare_rda.tsv", sep = "\t", header=0)
df.head()

,sample,SITE_ID,offset_rda,marker,trait,Trait_name,POP_SITE,POP_ID,SET,mean,sd,n,group
0,2209_PR,PR,0.082650,1000var5,Height,Height,2209_PR,2209,TEST,980.525258,302.158899,4.0,West
1,325_PR,PR,0.349505,1000var5,Height,Height,325_PR,325,TEST,1239.891145,121.743290,8.0,Central
2,4351_PR,PR,0.363508,1000var5,Height,Height,4351_PR,4351,TEST,1423.598605,68.068593,4.0,Central
3,4420_PR,PR,0.170629,1000var5,Height,Height,4420_PR,4420,TEST,678.153897,154.272486,7.0,West
4,6805_PR,PR,0.257768,1000var5,Height,Height,6805_PR,6805,TEST,1353.974749,229.855027,7.0,East


In [6]:
df[df['Trait_name'].isna()]

,sample,SITE_ID,offset_rda,marker,trait,Trait_name,POP_SITE,POP_ID,SET,mean,sd,n,group


# Bootstrapping

### Calculating rho and running bootstrap for samples n>=5 and ignoring cases when all offsets are = 0

In [7]:
df["marker"].drop_duplicates().values

array(['1000var5'], dtype=object)

In [8]:
import warnings
warnings.filterwarnings("ignore", message="An input array is constant")

nboots = 1000
i=0
results = []
for trait in df["Trait_name"].drop_duplicates().values:
#for trait in ["Height", "Biomass_Increment"]:
    print(trait)
    for marker in df["marker"].drop_duplicates().values:
        #print(marker)
        for dset in ["TRAIN", "TEST"]:
            #print(dset)
            for site in ['AC','ML','CH','PR']:
                #print(site)

                df_ = df[(df['SET']==dset) & (df['marker']==marker) & (df['Trait_name']==trait) & (df["SITE_ID"]==site)].reset_index(drop=True)
                size = df_.shape[0]
                offset_sum = df_['offset_rda'].sum()
                if (size > 4) & (offset_sum > 0):
                    rho, ci_low, ci_high = bootstrap_spearman(df_['offset_rda'], df_['mean'], nboot=nboots, conf=0.95, random_state=123+i)
                else:
                    rho, ci_low, ci_high = np.nan, np.nan, np.nan
                results.append((dset, marker, trait, site, rho, ci_low, ci_high, size, nboots))
                
                i+=1
dR = pd.DataFrame(results, columns = ["SET","marker","Trait_name","SITE_ID","rho","CI_low","CI_high","n_samples","n_boot"])

Height
Biomass_Increment
Biomass_Increment_1980
Biomass_Increment_1985
Biomass_Increment_1990
Biomass_Increment_1995
Biomass_Increment_2000
Biomass_Increment_2005
Biomass_Increment_2010
Biomass_Increment_2015
Average_Ring_Density
DBH
Rs
Rl
Rr
Rc


In [9]:
results
dR = pd.DataFrame(results, columns = ["SET","marker","trait","SITE_ID","rho","CI_low","CI_high","n_samples","n_boot"])
dR.head()

,SET,marker,trait,SITE_ID,rho,CI_low,CI_high,n_samples,n_boot
0,TRAIN,1000var5,Height,AC,-0.272727,-0.796242,0.459330,10,1000
1,TRAIN,1000var5,Height,ML,-0.226829,-0.501605,0.115703,40,1000
2,TRAIN,1000var5,Height,CH,-0.431660,-0.658287,-0.142780,36,1000
3,TRAIN,1000var5,Height,PR,0.327863,-0.020354,0.550391,26,1000
4,TEST,1000var5,Height,AC,-0.285873,-0.573522,0.066886,30,1000


In [10]:
dR.to_csv("02B_bootstrap_rda.tsv", sep="\t", header=True, index=False)

In [82]:
#df_ = df[(df['SET']=="TEST") & (df['marker']=='100LF') & (df['Trait_name']=='Biomass_Increment_1980') & (df["SITE_ID"]=='AC')].reset_index(drop=True)
#size = df_.shape[0]
#df_

In [79]:
df_['offset'].sum()

0.0